# Machine Learning for Applied Research

## 1: Problem Framing
In this notebook, we will cover:
- Translating a research question into a machine learning task
- A reusable problem-framing template: task type, target, features, metric, constraints
- Why a learned model can be useful even when an exact simulator already exists
- Loading and taking a first look at our running dataset
- Reading data with a "framing" mindset: which variables might matter, and why

### Introduction

Before training any model, a research question needs to be translated into a well-defined learning problem. This step is easy to rush past, but a poorly framed problem (e.g., the wrong target variable, an unrealistic evaluation metric, or features that will not actually be available when you need a prediction) cannot be fixed by a better model later.

A simple example: you might expect that a building with more window area needs more energy to heat in winter. But by how much, and how does that depend on the building's shape, height, and orientation? Throughout the next couple of notebooks, we will use exactly this problem as our running example: predicting how much heating and cooling energy a building needs, from its design.

### From a Research Question to a Learning Task

A useful template for framing covers five things: the task type (regression, classification, ...), the target variable, candidate features, an evaluation metric, and any constraints (small sample size, noisy measurements, imbalanced outcomes).

In [ ]:
# A simple framing template you can reuse for your own research question
problem_framing = {
    "research_question": "Can we predict how much heating energy a building needs, from its design?",
    "task_type": "regression",
    "target_variable": "heating load",
    "candidate_features": ["compactness", "surface area", "wall area", "roof area", "height", "orientation", "glazing area"],
    "evaluation_metric": "RMSE",
    "constraints": "only 768 building designs available, all from one simulation study"
}

for key, value in problem_framing.items():
    print(f"{key:20s}: {value}")

What happens if you pick the wrong evaluation metric before training a model? Consider a metric that rewards a model for being right on average but does not penalize occasional large errors: would that be a safe choice for a decision where a large error is much costlier than a small one?

### Why Learn a Model When an Exact Simulator Already Exists?

Our dataset for today was not measured in a lab. It was produced by running a building-energy simulation tool many times over different building designs. This raises a fair question: if a tool can already compute the exact heating and cooling load for any design, why bother training an approximate model on its outputs at all?

The answer is *speed*. Running a full simulation for one design might take seconds to minutes; running it thousands of times to explore a wide design space, or to plug into an optimization loop, can become impractical. A model trained on a set of precomputed examples, a **surrogate model**, can approximate the simulator's behavior almost instantly, trading a small amount of accuracy for a large amount of speed. The same idea applies whenever the "exact" source of truth is expensive: a slow simulation, a costly lab measurement, or a time-consuming field experiment.

### Meet the Dataset: Building Energy Efficiency

We will work with the Energy Efficiency dataset from the UCI Machine Learning Repository. It contains 768 building designs, each simulated across 8 design parameters, with two possible targets: heating load and cooling load. We will focus on heating load for this notebook.

In [ ]:
# Install the UCI ML Repository helper package (only needs to run once per session)
!pip install ucimlrepo --quiet

#### 1.1 Exercises:
> (a) Import `fetch_ucirepo` from the `ucimlrepo` package you just installed, and import `pandas`.
>
> (b) Fetch the Energy Efficiency dataset using `fetch_ucirepo(id=242)` and store it as `energy`.
>
> (c) Split the result into `X` (the features) and `y` (the targets), and print the column names of each. We can access the `.features` and `.targets` from `energy.data`
>
> (d) We only want the first target column (heating load) for this notebook. Store its name in a variable `target_col`, then build a single dataframe `df` that combines `X` with just that one target column. Print the shape of `df` and preview its first rows.

In [ ]:
# Your solution:




##### Solution:


In [ ]:
from ucimlrepo import fetch_ucirepo
import pandas as pd

energy = fetch_ucirepo(id=242)

X = energy.data.features
y = energy.data.targets

X.columns = ["Relative Compactness",
             "Surface Area",
             "Wall Area",
             "Roof Area",
             "Overall Height",
             "Orientation",
             "Glazing Area",
             "Glazing Area Distribution"]

y.columns = ["Heating Load", "Cooling Load"]

print(f"Feature columns: {X.columns.tolist()}")
print(f"Target columns:  {y.columns.tolist()}")

target_col = y.columns[0]   # Heating Load
df = pd.concat([X, y[[target_col]]], axis=1)

print(f"\nShape: {df.shape}")
df.head()

#### Dataset

Each row is one simulated building design. The columns are:

| Column | Meaning |
|---|---|
| Relative Compactness | how compact the building's shape is (closer to a cube = more compact) |
| Surface Area | total exterior surface area (m²) |
| Wall Area | total wall area (m²) |
| Roof Area | total roof area (m²) |
| Overall Height | building height (m) |
| Orientation | which compass direction the building mainly faces (coded 2-5) |
| Glazing Area | proportion of exterior surface that is glass (windows) |
| Glazing Area Distribution | how the glazing is distributed across the building's sides |
| Heating Load | **target used in this notebook**: energy needed to heat the building |

Note that the dataset also includes a Cooling Load column, which we are deliberately not using here. This is a reminder that "which target?" is itself a framing decision, not just "which features?".

In [ ]:
# A first framing-informed look: which features move together with the target?
correlations = df.corr(numeric_only=True)[target_col].sort_values(ascending=False)
print(correlations)

Which feature would you have guessed matters most for how much heating a building needs? Does the correlation table support or contradict your intuition?

#### 1.2 Exercises:
> (a) Later this week you will work with a second dataset: a factory records sensor readings (air temperature, rotational speed, torque, tool wear) from its machines and wants to predict whether a machine will fail before it happens. Write a `problem_framing` dictionary for this scenario, following the template above.
>
> (b) Identify the target variable and at least three candidate features in your dictionary.
>
> (c) In one sentence, name one constraint this dataset is likely to have that the building dataset does not (hint: think about how often machines actually fail in practice).

In [ ]:
# Your solution:




##### Solution:


In [ ]:
problem_framing_maintenance = {
    "research_question": "Can we predict machine failure from sensor readings?",
    "task_type": "classification (binary)",
    "target_variable": "machine failure (0/1)",
    "candidate_features": ["air temperature", "process temperature", "rotational speed", "torque", "tool wear"],
    "evaluation_metric": "precision/recall rather than accuracy alone",
    "constraints": "failures are rare - the two classes are strongly imbalanced"
}

for key, value in problem_framing_maintenance.items():
    print(f"{key:20s}: {value}")

# (c) Unlike the building dataset, where heating-load values are spread
# fairly evenly across a range, failures in this scenario are rare events.
# Most machines will not fail, so the two outcome classes are imbalanced,
# a constraint the building dataset does not have.

#### 1.3 Exercises:
> (a) Using the `df` dataframe loaded above, confirm the number of rows and columns, and check whether there are any missing values.
>
> (b) Compute the correlation between `Overall Height` and the target column (`target_col`). Is the direction (positive or negative) what you expected?
>
> (c) Identify the two features most strongly correlated with the target from `correlations`, in either direction.

In [ ]:
# Your solution:




##### Solution:


In [ ]:
# (a)
print(f"Rows, columns: {df.shape}")
print(f"Missing values per column:\n{df.isnull().sum()}")

# (b)
height_corr = df["Overall Height"].corr(df[target_col])
print(f"\nCorrelation between Overall Height and {target_col}: {height_corr:.3f}")
# Positive and strong, as expected: taller buildings (for a similar
# footprint) tend to need more energy to heat.

# (c)
top_two = correlations.drop(target_col).abs().sort_values(ascending=False).head(2)
print(f"\nMost strongly correlated features:\n{top_two}")

#### Challenge (optional):
> Repeat the correlation analysis using `Cooling Load` (the dataset's second target) instead of `Heating Load`. Are the most important features the same for both targets? Can you think of an architectural reason why heating and cooling might depend differently on a building's design?

In [ ]:
# Your solution:




##### Solution:


In [ ]:
cooling_col = y.columns[1]   # Cooling Load
df_cooling = pd.concat([X, y[[cooling_col]]], axis=1)

cooling_correlations = df_cooling.corr(numeric_only=True)[cooling_col].sort_values(ascending=False)
print(cooling_correlations)

print("\nHeating Load - top features:")
print(correlations.drop(target_col).abs().sort_values(ascending=False).head(3))

print("\nCooling Load - top features:")
print(cooling_correlations.drop(cooling_col).abs().sort_values(ascending=False).head(3))

# Compare the two rankings above. In building energy studies, glazing area
# and orientation often turn out to matter more for cooling load than for
# heating load, since solar heat gain through windows drives cooling
# demand more directly, but check whether that holds for your output.

### Conclusion

This notebook covered the first step of any applied ML project, **structured problem framing**: turning a research question into a task type, target, feature set, metric, and constraints.

We discussed **surrogate modeling**: one concrete reason to train a model even when an exact (but expensive) source of truth already exists.

We know now to **read our data with intent**: a first pass over a new dataset should already be informed by the framing, not just exploratory for its own sake.

We introduced the **Building Energy Efficiency dataset**: our running example for the next couple of notebooks.

Notebook 2 continues with this same dataset for a closer look at the data itself: exploratory data analysis, before we get to supervised learning foundations in Notebook 3.